In [32]:
#import libraries
import pandas as pd
import numpy as np
import pyarrow

In [33]:
#load dataset
df = pd.read_csv("../data/raw/home_credit/application_train.csv")

#print row count, column list, data types, and null counts (top 15 by null count)
print("Row count: ", df.shape[0])
print("\nColumn list: ", df.columns)
print("\nData Types: ", df.dtypes)
print("\nNull Counts: ", df.isnull().sum().sort_values(ascending=False).head(15))

Row count:  307511

Column list:  Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='object', length=122)

Data Types:  SK_ID_CURR                      int64
TARGET                          int64
NAME_CONTRACT_TYPE             object
CODE_GENDER                    object
FLAG_OWN_CAR                   object
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     float64
AMT_REQ_CREDIT_BUREAU_WEEK    float64
AMT_REQ_CREDIT_BUREAU_MON     float64
AMT_REQ_CREDIT_BUREAU_QRT     float64
AMT_REQ_CREDIT_BUREAU_YEAR    float64
Length: 122, d

In [34]:
#CLEANING
#drop columns with more than 40% missing
missing_percent = df.isnull().mean()

cols_to_drop = missing_percent[missing_percent > 0.4].index.tolist()

print("Columns dropped (>40% missing):")
for col in cols_to_drop:
    print(col)

df = df.drop(columns=cols_to_drop)
print("\nNumber of columns dropped:", len(cols_to_drop))

Columns dropped (>40% missing):
OWN_CAR_AGE
EXT_SOURCE_1
APARTMENTS_AVG
BASEMENTAREA_AVG
YEARS_BEGINEXPLUATATION_AVG
YEARS_BUILD_AVG
COMMONAREA_AVG
ELEVATORS_AVG
ENTRANCES_AVG
FLOORSMAX_AVG
FLOORSMIN_AVG
LANDAREA_AVG
LIVINGAPARTMENTS_AVG
LIVINGAREA_AVG
NONLIVINGAPARTMENTS_AVG
NONLIVINGAREA_AVG
APARTMENTS_MODE
BASEMENTAREA_MODE
YEARS_BEGINEXPLUATATION_MODE
YEARS_BUILD_MODE
COMMONAREA_MODE
ELEVATORS_MODE
ENTRANCES_MODE
FLOORSMAX_MODE
FLOORSMIN_MODE
LANDAREA_MODE
LIVINGAPARTMENTS_MODE
LIVINGAREA_MODE
NONLIVINGAPARTMENTS_MODE
NONLIVINGAREA_MODE
APARTMENTS_MEDI
BASEMENTAREA_MEDI
YEARS_BEGINEXPLUATATION_MEDI
YEARS_BUILD_MEDI
COMMONAREA_MEDI
ELEVATORS_MEDI
ENTRANCES_MEDI
FLOORSMAX_MEDI
FLOORSMIN_MEDI
LANDAREA_MEDI
LIVINGAPARTMENTS_MEDI
LIVINGAREA_MEDI
NONLIVINGAPARTMENTS_MEDI
NONLIVINGAREA_MEDI
FONDKAPREMONT_MODE
HOUSETYPE_MODE
TOTALAREA_MODE
WALLSMATERIAL_MODE
EMERGENCYSTATE_MODE

Number of columns dropped: 49


In [35]:
#impute median (numeric cols)
num_cols = df.select_dtypes(include=np.number).columns
num_missing_before = df[num_cols].isna().sum().sum()

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
num_missing_after = df[num_cols].isna().sum().sum()

print("Numerical columns missing before:", num_missing_before)
print("Numerical columns missing after:", num_missing_after)

Numerical columns missing before: 315116
Numerical columns missing after: 0


In [36]:
#impute mode (categorical cols)
cat_cols = df.select_dtypes(exclude=np.number).columns
cat_missing_before = df[cat_cols].isna().sum().sum()

df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])
cat_missing_after = df[cat_cols].isna().sum().sum()

print("Categorical columns missing before:", cat_missing_before)
print("Categorical columns missing after:", cat_missing_after)


Categorical columns missing before: 97683
Categorical columns missing after: 0


In [37]:
#Remove rows where AMT_INCOME TOTAL is an extreme outlier (beyond 99.5th percentile)
threshold = df["AMT_INCOME_TOTAL"].quantile(0.995)
rows_before = df.shape[0]

df = df[df["AMT_INCOME_TOTAL"] < threshold]
rows_after = df.shape[0]

print("Rows removed with extreme income outliers: ", rows_before - rows_after)


Rows removed with extreme income outliers:  1568


In [38]:
#Remove rows where DAYS_EMPLOYED has anomalous value 365243
rows_before1 = df.shape[0]
df = df[df["DAYS_EMPLOYED"] != 365243]

rows_after1 = df.shape[0]

print("Rows removed with anomalous value of 365243: ", rows_before1 - rows_after1)


Rows removed with anomalous value of 365243:  55288


In [39]:
#feature engineering
df["credit_income_ratio"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["annuity_income_ratio"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["credit_term"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]
df["days_employed_ratio"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
df["income_per_family"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

In [40]:
#Encode categorical variables using one-hot encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [41]:
#Save the curated dataset
df.to_parquet("../data/curated/home_credit_curated.parquet")